# LIP-TRAC: final training run

This notebook reproduces the configuration behind the numbers in the LIP-TRAC paper: the
12.9M-parameter Conv3D + bidirectional-GRU model, trained with CTC on mouth crops from a
1,139-clip sample of LRS2 `main`, batches of 4, a 60/40 split, and a staged Adam schedule.

It is a clean reconstruction of `Lipreading.ipynb` / `LipreadingMOREEPOCHS.ipynb` (the runs that
saved `models/FINAL_LIPMODEL.weights.h5`). Those notebooks were interrupted and resumed several
times with `epochs=300` each time, so **the number of epochs per stage below is the one thing
that cannot be read off the surviving code**; `STAGES` is set to sum to the 300 epochs reported
in the paper. The data pipeline and the model are copied unchanged. Nothing in this notebook has
been executed in the repository; outputs are intentionally absent.

Requirements: `pip install -r ../../requirements.txt`, a copy of LRS2 (not redistributable; see the
README), and the `trainFiles1`-style folder of `.mp4` / `.txt` pairs produced by
`notebooks/data/trainFileGen.ipynb`.

In [ ]:
import os
import csv
import random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
from jiwer import wer, cer
from matplotlib import pyplot as plt

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Paths. DATA_DIR holds pairs <clip>.mp4 / <clip>.txt copied from LRS2 "main"
# (see notebooks/data/trainFileGen.ipynb). LRS2 itself is not in this repository.
DATA_DIR = Path(os.environ.get("LIPTRAC_DATA", "../../data/trainFiles1"))
CASCADE_PATH = Path("../../assets/haarcascade_frontalface_default.xml")
WEIGHTS_PATH = Path("../../models/FINAL_LIPMODEL.weights.h5")
OUT_DIR = Path("runs/final_run")

# Data and split. The final run used 1,139 clips, batches of 4, and a 60/40 split
# taken over batches (171 training batches = 684 clips; 114 held-out batches = 455 clips,
# the last batch holding 3). The original shuffle was unseeded, so the exact clips in each
# half of the reported run cannot be reconstructed; SEED fixes a split for this notebook.
SEED = 1139
BATCH_SIZE = 4
TRAIN_FRACTION = 0.6

# Input geometry (fixed by the model: 290 frames of 40 x 120 grayscale mouth crops).
IMAGE_SIZE = (40, 120)   # (height, width)
MAX_FRAMES = 290         # 2 * MAX_CHARS, as in the original notebooks
MAX_CHARS = 145

## Data pipeline

In [ ]:
# Characters -> integers and back. Index 0 is the padding/OOV token; the Dense layer adds
# one more class for the CTC blank, giving 29 classes.
VOCAB = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ ")
char_to_num = tf.keras.layers.StringLookup(vocabulary=VOCAB, oov_token="")
num_to_char = tf.keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)
NUM_CLASSES = char_to_num.vocabulary_size() + 1

face_cascade = cv2.CascadeClassifier(str(CASCADE_PATH))
assert not face_cascade.empty(), f"could not load {CASCADE_PATH}"
last_known_crop = (0, 0, 160, 150)


def detect_faces(gray):
    return face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))


def crop_mouth(frame):
    """Fixed crop relative to the largest detected face; reuses the last crop when no face is found.

    Rows: from 65% of the face height to the bottom of the box.
    Columns: from x + 0.05 * w to 0.95 * (x + w). This is exactly what the training notebooks
    and pi/cropFuncs.py do; note that the right edge is 95% of the right coordinate, not
    x + 0.95 * w."""
    global last_known_crop
    rects = detect_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    largest = (0, 0, 0, 0)
    for (x, y, w, h) in rects:
        if w * h > largest[2] * largest[3]:
            largest = (x, y, w, h)
    if largest == (0, 0, 0, 0):
        largest = last_known_crop
    last_known_crop = largest
    x, y, w, h = largest
    return frame[y + int(0.65 * h): y + h, x + int(0.05 * w): int(0.95 * (x + w))]


def load_video(path):
    """Mouth crops for every frame, resized to IMAGE_SIZE, grayscale, standardized per clip."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    first_shape = None
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
        ok, frame = cap.read()
        if not ok or frame is None or frame.shape[0] == 0:
            continue
        if first_shape is None:
            first_shape = frame.shape
        if frame.shape != first_shape:
            continue
        mouth = crop_mouth(frame)
        mouth = cv2.resize(mouth, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
        frames.append(tf.image.rgb_to_grayscale(mouth))
    cap.release()
    # Same arithmetic as the original notebooks: the mean is taken over the uint8 frames
    # (so it is truncated to an integer) and the standard deviation in float32.
    mean = tf.math.reduce_mean(frames, keepdims=True)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32), keepdims=True)
    frames = tf.cast(frames, tf.float32)
    return (frames - tf.cast(mean, tf.float32)) / std


def number_to_words(num):
    if num == 0:
        return "ZERO"
    ones = ["", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
    tens = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
    teens = ["ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
    words = ""
    if num >= 1000:
        words += ones[num // 1000] + " thousand "
        num %= 1000
    if num >= 100:
        words += ones[num // 100] + " hundred "
        num %= 100
    if 10 <= num <= 19:
        words += teens[num - 10] + " "
        num = 0
    elif num >= 20:
        words += tens[num // 10] + " "
        num %= 10
    if 1 <= num <= 9:
        words += ones[num] + " "
    return words.strip().upper()


def load_transcript(path):
    """LRS2 transcript files start with a line 'Text:  THE WORDS ...'. Numerals become words."""
    with open(path, "r") as f:
        words = f.readline().split()[1:]
    words = [number_to_words(int(w)) if w.isnumeric() else w for w in words]
    text = " ".join(" ".join(words).split()).upper()
    return char_to_num(tf.strings.unicode_split(text, input_encoding="UTF-8"))


def load_clip(path_tensor):
    stem = Path(bytes.decode(path_tensor.numpy())).stem
    return load_video(DATA_DIR / f"{stem}.mp4"), load_transcript(DATA_DIR / f"{stem}.txt")


def process_clip(path):
    return tf.py_function(load_clip, [path], (tf.float32, tf.int64))


def decode_labels(label_ids):
    return tf.strings.reduce_join(num_to_char(label_ids)).numpy().decode("utf-8").strip()

## Split

In [ ]:
clip_paths = sorted(str(p) for p in DATA_DIR.glob("*.mp4"))
assert clip_paths, f"no clips found in {DATA_DIR.resolve()}"
print("clips:", len(clip_paths))

dataset = tf.data.Dataset.from_tensor_slices(clip_paths)
dataset = dataset.shuffle(len(clip_paths), seed=SEED, reshuffle_each_iteration=False)
dataset = dataset.map(process_clip)
dataset = dataset.padded_batch(BATCH_SIZE, padded_shapes=([MAX_FRAMES, None, None, None], [MAX_CHARS]))
dataset = dataset.prefetch(tf.data.AUTOTUNE)

num_batches = len(dataset)
num_train_batches = int(num_batches * TRAIN_FRACTION)
train_ds = dataset.take(num_train_batches)
test_ds = dataset.skip(num_train_batches)
print("batches:", num_batches, " train batches:", num_train_batches, " held-out batches:", num_batches - num_train_batches)

# Record the split parameters next to the run.
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "split_seed.txt", "w") as f:
    f.write(f"seed={SEED} batch_size={BATCH_SIZE} train_fraction={TRAIN_FRACTION} clips={len(clip_paths)}\n")

## Model and loss

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, TimeDistributed, Flatten, Bidirectional, GRU, Dropout, Dense

INPUT_SHAPE = (MAX_FRAMES, IMAGE_SIZE[0], IMAGE_SIZE[1], 1)


def create_model(gru_units=256):
    return Sequential([
        Conv3D(32, 3, input_shape=INPUT_SHAPE, padding="same", activation="relu", name="conv1"),
        MaxPooling3D((1, 2, 2), name="maxPool1"),
        Conv3D(64, 3, padding="same", activation="relu", name="conv2"),
        MaxPooling3D((1, 2, 2), name="maxPool2"),
        Conv3D(96, 3, padding="same", activation="relu", name="conv3"),
        MaxPooling3D((1, 2, 2), name="maxPool3"),
        TimeDistributed(Flatten()),
        Bidirectional(GRU(gru_units, kernel_initializer="orthogonal", return_sequences=True)),
        Dropout(0.5),
        Bidirectional(GRU(gru_units, kernel_initializer="orthogonal", return_sequences=True)),
        Dropout(0.5),
        Dense(NUM_CLASSES, kernel_initializer="he_normal", activation="softmax"),
    ])


def ctc_loss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], "int64")
    input_len = tf.cast(tf.shape(y_pred)[1], "int64") * tf.ones((batch_len, 1), "int64")
    label_len = tf.cast(tf.shape(y_true)[1], "int64") * tf.ones((batch_len, 1), "int64")
    return tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_len, label_len)


model = create_model(256)
model.summary()   # expected total: 12,875,325 parameters

## Decoding and metrics

In [ ]:
def decode_batch(y_pred, greedy=False, beam_width=100):
    """CTC decoding of a batch of per-frame distributions. The original evaluation code passed the
    full padded length (MAX_FRAMES) for every clip and used Keras' beam search (beam width 100);
    the Raspberry Pi scripts decode greedily."""
    lengths = [MAX_FRAMES] * int(y_pred.shape[0])
    decoded = tf.keras.backend.ctc_decode(y_pred, input_length=lengths, greedy=greedy, beam_width=beam_width, top_paths=1)[0][0]
    return [decode_labels(row) for row in decoded.numpy()]


def evaluate(model_or_predict, ds, greedy=False):
    """Corpus-level WER and CER over every clip in ds, plus the per-clip pairs."""
    refs, hyps = [], []
    for frames, labels in ds:
        y_pred = model_or_predict(frames)
        hyps.extend(decode_batch(y_pred, greedy=greedy))
        refs.extend(decode_labels(row) for row in labels.numpy())
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "clips": len(refs)}, list(zip(refs, hyps))

## Learning-rate schedule

In [ ]:
# Adam learning-rate schedule, as run in the surviving notebooks (four compile/fit stages).
#   decay=None            constant rate
#   decay="after_epoch_5" multiply by exp(-0.1) each epoch from the sixth epoch of the stage
#   decay="every_epoch"   multiply by exp(-0.1) every epoch
# EPOCHS PER STAGE ARE A PLACEHOLDER (they sum to the 300 epochs reported in the paper).
STAGES = [
    {"lr": 2e-4, "epochs": 75, "decay": None},
    {"lr": 6e-5, "epochs": 75, "decay": None},
    {"lr": 2e-5, "epochs": 75, "decay": "after_epoch_5"},
    {"lr": 7e-6, "epochs": 75, "decay": "every_epoch"},
]
assert sum(s["epochs"] for s in STAGES) == 300


def make_scheduler(kind):
    if kind is None:
        return None
    if kind == "after_epoch_5":
        return tf.keras.callbacks.LearningRateScheduler(lambda epoch, lr: lr if epoch < 5 else lr * float(tf.math.exp(-0.1)))
    if kind == "every_epoch":
        return tf.keras.callbacks.LearningRateScheduler(lambda epoch, lr: lr * float(tf.math.exp(-0.1)))
    raise ValueError(kind)

## Callbacks

In [ ]:
class HeldOutMetrics(tf.keras.callbacks.Callback):
    """After every epoch: beam-search decode the held-out clips and log WER, CER and the
    accuracies (1 - WER, 1 - CER) that the paper's training curves show."""

    def __init__(self, ds, log_path):
        super().__init__()
        self.ds = ds
        self.log_path = Path(log_path)
        self.global_epoch = 0
        if not self.log_path.exists():
            with open(self.log_path, "w", newline="") as f:
                csv.writer(f).writerow(["epoch", "stage", "lr", "loss", "val_loss", "wer", "cer", "word_acc", "char_acc"])
        self.stage = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.global_epoch += 1
        metrics, _ = evaluate(self.model.predict, self.ds, greedy=False)
        with open(self.log_path, "a", newline="") as f:
            csv.writer(f).writerow([
                self.global_epoch, self.stage, float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)),
                logs.get("loss"), logs.get("val_loss"), metrics["wer"], metrics["cer"],
                1 - metrics["wer"], 1 - metrics["cer"],
            ])
        print(f"epoch {self.global_epoch}: held-out WER {metrics['wer']:.3f}  CER {metrics['cer']:.3f}")


class ShowExample(tf.keras.callbacks.Callback):
    """Print one held-out batch of reference / prediction pairs each epoch, as the original notebooks did."""

    def __init__(self, ds):
        super().__init__()
        self.batch = next(iter(ds))

    def on_epoch_end(self, epoch, logs=None):
        frames, labels = self.batch
        hyps = decode_batch(self.model.predict(frames, verbose=0))
        for ref_ids, hyp in zip(labels.numpy(), hyps):
            print("ref:", decode_labels(ref_ids))
            print("hyp:", hyp)
            print("-" * 60)


metrics_cb = HeldOutMetrics(test_ds, OUT_DIR / "training_history.csv")
example_cb = ShowExample(test_ds)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(str(OUT_DIR / "checkpoint_epoch{epoch:03d}.weights.h5"), save_weights_only=True, save_freq="epoch")

## Train

In [ ]:
for i, stage in enumerate(STAGES):
    print(f"=== stage {i}: lr={stage['lr']}  epochs={stage['epochs']}  decay={stage['decay']}")
    metrics_cb.stage = i
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=stage["lr"]), loss=ctc_loss)
    callbacks = [checkpoint_cb, example_cb, metrics_cb]
    scheduler = make_scheduler(stage["decay"])
    if scheduler is not None:
        callbacks.append(scheduler)
    model.fit(train_ds, validation_data=test_ds, epochs=stage["epochs"], callbacks=callbacks)

WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save_weights(str(WEIGHTS_PATH))
print("saved", WEIGHTS_PATH)

## Training curves

In [ ]:
# Training curves from the per-epoch log (loss, word accuracy, character accuracy on the held-out clips).
rows = list(csv.DictReader(open(OUT_DIR / "training_history.csv")))
epochs = [int(r["epoch"]) for r in rows]

fig_dir = OUT_DIR / "figures"
fig_dir.mkdir(exist_ok=True)
for key, label, fname in [("loss", "CTC loss", "loss_curve.png"), ("word_acc", "Word accuracy", "word_acc_curve.png"), ("char_acc", "Character accuracy", "char_acc_curve.png")]:
    plt.figure(figsize=(5, 3.2))
    if key == "loss":
        plt.plot(epochs, [float(r["loss"]) for r in rows], label="training")
        plt.plot(epochs, [float(r["val_loss"]) for r in rows], label="held-out")
        plt.legend()
    else:
        plt.plot(epochs, [float(r[key]) for r in rows])
    plt.xlabel("Epoch")
    plt.ylabel(label)
    plt.tight_layout()
    plt.savefig(fig_dir / fname, dpi=200)
    plt.show()

## Held-out WER and CER

In [ ]:
# Final numbers on the held-out clips with the full-precision Keras model.
beam_metrics, pairs = evaluate(model.predict, test_ds, greedy=False)
greedy_metrics, _ = evaluate(model.predict, test_ds, greedy=True)
print("beam search (beam width 100):", beam_metrics)
print("greedy:                      ", greedy_metrics)

with open(OUT_DIR / "heldout_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["reference", "prediction"])
    w.writerows(pairs)